# Schema Contracts for CDC Pipeline

This notebook defines expected schema contracts for all tables in the dvdrental CDC pipeline.
These contracts are used for schema drift detection and validation.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    LongType, DecimalType, TimestampType, BooleanType
)

In [ ]:
# ============================================================================
# BRONZE LAYER SCHEMAS (Raw Kafka CDC Events)
# All three tables share the same Bronze envelope schema.
# ============================================================================

_BRONZE_ENVELOPE = StructType([
    StructField("topic",          StringType(),    False),
    StructField("partition",      IntegerType(),   False),
    StructField("offset",         LongType(),      False),
    StructField("kafka_timestamp",TimestampType(), True),
    StructField("message_key",    StringType(),    True),
    StructField("value",          StringType(),    True),
    StructField("source_schema",  StringType(),    True),
    StructField("table_name",     StringType(),    True),
    StructField("ingested_at",    TimestampType(), False),
])

BRONZE_RENTAL_SCHEMA  = _BRONZE_ENVELOPE
BRONZE_FILM_SCHEMA    = _BRONZE_ENVELOPE
BRONZE_PAYMENT_SCHEMA = _BRONZE_ENVELOPE

In [ ]:
# ============================================================================
# SILVER LAYER SCHEMAS (Cleansed, current-state)
# ============================================================================

SILVER_RENTAL_SCHEMA = StructType([
    StructField("rental_id",       IntegerType(),   False),
    StructField("rental_date",     TimestampType(), True),
    StructField("inventory_id",    IntegerType(),   True),
    StructField("customer_id",     IntegerType(),   True),
    StructField("return_date",     TimestampType(), True),
    StructField("staff_id",        IntegerType(),   True),
    StructField("last_update",     TimestampType(), True),
    StructField("last_inserted_dt",TimestampType(), True),
    StructField("last_updated_dt", TimestampType(), False),
])

SILVER_FILM_SCHEMA = StructType([
    StructField("film_id",          IntegerType(),      False),
    StructField("title",            StringType(),       False),
    StructField("description",      StringType(),       True),
    StructField("release_year",     IntegerType(),      True),
    StructField("language_id",      IntegerType(),      True),
    StructField("rental_duration",  IntegerType(),      True),
    StructField("rental_rate",      DecimalType(4, 2),  True),
    StructField("length",           IntegerType(),      True),
    StructField("replacement_cost", DecimalType(5, 2),  True),
    StructField("rating",           StringType(),       True),
    StructField("last_update",      TimestampType(),    True),
    StructField("last_inserted_dt", TimestampType(),    True),
    StructField("last_updated_dt",  TimestampType(),    False),
])

SILVER_PAYMENT_SCHEMA = StructType([
    StructField("payment_id",      IntegerType(),     False),
    StructField("customer_id",     IntegerType(),     True),
    StructField("staff_id",        IntegerType(),     True),
    StructField("rental_id",       IntegerType(),     True),
    StructField("amount",          DecimalType(5, 2), True),
    StructField("payment_date",    TimestampType(),   True),
    StructField("last_inserted_dt",TimestampType(),   True),
    StructField("last_updated_dt", TimestampType(),   False),
])

In [ ]:
# ============================================================================
# GOLD LAYER SCHEMAS (Business-Ready)
# ============================================================================

GOLD_RENTAL_SCHEMA = StructType([
    StructField("rental_id",       IntegerType(),   False),
    StructField("rental_date",     TimestampType(), True),
    StructField("inventory_id",    IntegerType(),   True),
    StructField("customer_id",     IntegerType(),   True),
    StructField("return_date",     TimestampType(), True),
    StructField("staff_id",        IntegerType(),   True),
    StructField("rental_status",   StringType(),    False),  # open / returned
    StructField("total_paid",      DecimalType(10, 2), True),
    StructField("last_updated_dt", TimestampType(), False),
])

GOLD_FILM_SCHEMA = StructType([
    StructField("film_id",           IntegerType(),     False),
    StructField("title",             StringType(),      False),
    StructField("description",       StringType(),      True),
    StructField("release_year",      IntegerType(),     True),
    StructField("rental_duration",   IntegerType(),     True),
    StructField("rental_rate",       DecimalType(4, 2), True),
    StructField("rental_rate_tier",  StringType(),      False),  # budget / standard / premium
    StructField("length",            IntegerType(),     True),
    StructField("replacement_cost",  DecimalType(5, 2), True),
    StructField("rating",            StringType(),      True),
    StructField("last_update",       TimestampType(),   True),
])

In [ ]:
# ============================================================================
# SCHEMA CONTRACT REGISTRY
# ============================================================================

SCHEMA_CONTRACTS = {
    # Bronze layer
    "bronze.rental":  BRONZE_RENTAL_SCHEMA,
    "bronze.film":    BRONZE_FILM_SCHEMA,
    "bronze.payment": BRONZE_PAYMENT_SCHEMA,
    # Silver layer
    "silver.rental":  SILVER_RENTAL_SCHEMA,
    "silver.film":    SILVER_FILM_SCHEMA,
    "silver.payment": SILVER_PAYMENT_SCHEMA,
    # Gold layer
    "gold.rental":    GOLD_RENTAL_SCHEMA,
    "gold.film":      GOLD_FILM_SCHEMA,
}

SCHEMA_POLICIES = {
    "bronze": {"policy": "additive_only", "alert_channel": "all", "block_on_critical": True},
    "silver": {"policy": "strict",        "alert_channel": "all", "block_on_critical": True},
    "gold":   {"policy": "strict",        "alert_channel": "all", "block_on_critical": True},
}


def get_schema_contract(contract_name: str) -> StructType:
    if contract_name not in SCHEMA_CONTRACTS:
        available = ", ".join(SCHEMA_CONTRACTS.keys())
        raise KeyError(
            f"Schema contract '{contract_name}' not found. "
            f"Available contracts: {available}"
        )
    return SCHEMA_CONTRACTS[contract_name]


def get_layer_policy(layer: str) -> dict:
    if layer not in SCHEMA_POLICIES:
        raise KeyError(f"Policy for layer '{layer}' not found")
    return SCHEMA_POLICIES[layer]

## Usage Example

```python
%run ./helpers/NB_schema_contracts

rental_contract = get_schema_contract('silver.rental')
silver_policy = get_layer_policy('silver')
```